# Report NaNs

Scans all CSVs in `experimental_results/` and reports missing-value rates in `model_response` and `model_response_pole` (where applicable) per model, per experiment, and in aggregate.

In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

RESULTS_ROOT = Path('..') / 'experimental_results'

ABSOLUTE_COLS = ['model_response']
COMPARATIVE_COLS = ['model_response', 'model_response_pole']

COLUMNS_BY_EXPERIMENT_TYPE = {
    'absolute_experiment': ABSOLUTE_COLS,
    'comparative_experiment_with_ground_truth': COMPARATIVE_COLS,
    'comparative_experiment_with_ground_truth_and_multiple_choices': COMPARATIVE_COLS,
    'comparative_experiment_without_ground_truth': COMPARATIVE_COLS,
    'comparative_experiment_without_ground_truth_and_multiple_choices': COMPARATIVE_COLS,
    'unblind_experiment': COMPARATIVE_COLS,
}

In [ ]:
def _extract_model_name(stem: str) -> str:
    """Strip reasoning-effort suffix from filename stem to recover the model name."""
    return re.sub(r'_reasoning_effort_(low|medium|high)$', '', stem)


records = []

for experiment_name_dir in sorted(RESULTS_ROOT.iterdir()):
    if not experiment_name_dir.is_dir():
        continue
    experiment_name = experiment_name_dir.name

    for experiment_type_dir in sorted(experiment_name_dir.iterdir()):
        if not experiment_type_dir.is_dir():
            continue
        experiment_type = experiment_type_dir.name
        cols_to_check = COLUMNS_BY_EXPERIMENT_TYPE.get(experiment_type, ABSOLUTE_COLS)

        for csv_path in sorted(experiment_type_dir.glob('*.csv')):
            model_name = _extract_model_name(csv_path.stem)
            try:
                df = pd.read_csv(csv_path)
            except Exception as e:
                print(f'  ERROR reading {csv_path}: {e}')
                continue

            total_rows = len(df)
            for col in cols_to_check:
                if col not in df.columns:
                    # Column absent for this experiment type – not a NaN, just skip
                    continue
                nan_count = int(df[col].isna().sum())
                records.append({
                    'experiment_name': experiment_name,
                    'experiment_type': experiment_type,
                    'model_name': model_name,
                    'column': col,
                    'total_rows': total_rows,
                    'nan_count': nan_count,
                    'nan_rate': nan_count / total_rows if total_rows > 0 else np.nan,
                })

df_detail = pd.DataFrame(records)
print(f'Total records collected: {len(df_detail)}')
print(f'Experiments found: {df_detail["experiment_name"].nunique()}')
print(f'Models found: {df_detail["model_name"].nunique()}')

## Per-model × per-experiment: `model_response` NaN rates

In [ ]:
def make_pivot(df_detail, column_name):
    """Return a styled pivot: rows=models, columns=experiment_name/experiment_type, values=nan_rate%."""
    subset = df_detail[df_detail['column'] == column_name].copy()
    if subset.empty:
        print(f'No data for column: {column_name}')
        return

    subset['experiment_label'] = subset['experiment_name'] + '\n' + subset['experiment_type']
    pivot = subset.pivot_table(
        index='model_name',
        columns='experiment_label',
        values='nan_rate',
        aggfunc='mean',
    )
    pivot = pivot * 100  # convert to percent

    # Add a row-level mean for sorting
    pivot['__mean__'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('__mean__', ascending=False)
    pivot = pivot.drop(columns='__mean__')

    def color_nonzero(val):
        if pd.isna(val):
            return 'color: lightgrey'
        if val > 0:
            intensity = min(int(val * 2.55), 200)
            return f'background-color: rgba(220, 50, 50, {val/100:.2f}); color: white'
        return ''

    styled = (
        pivot.style
        .format('{:.1f}%', na_rep='—')
        .applymap(color_nonzero)
        .set_caption(f'NaN rate (%) — {column_name}')
        .set_table_styles([{'selector': 'th', 'props': [('font-size', '10px'), ('text-align', 'center')]}])
    )
    display(styled)


make_pivot(df_detail, 'model_response')

## Per-model × per-experiment: `model_response_pole` NaN rates

Only comparative and unblind experiments include this column.

In [ ]:
make_pivot(df_detail, 'model_response_pole')

## Overall NaN rate per model

In [ ]:
def overall_by_model(df_detail):
    rows = []
    for model_name, grp in df_detail.groupby('model_name'):
        row = {'model_name': model_name}
        for col in ['model_response', 'model_response_pole']:
            sub = grp[grp['column'] == col]
            if sub.empty:
                row[f'{col}_total'] = 0
                row[f'{col}_nan_count'] = 0
                row[f'{col}_nan_rate'] = np.nan
            else:
                total = sub['total_rows'].sum()
                nan_count = sub['nan_count'].sum()
                row[f'{col}_total'] = total
                row[f'{col}_nan_count'] = nan_count
                row[f'{col}_nan_rate'] = nan_count / total * 100 if total > 0 else np.nan
        rows.append(row)

    df_out = pd.DataFrame(rows).set_index('model_name')
    df_out = df_out.sort_values('model_response_nan_rate', ascending=False)
    display_cols = [
        'model_response_total', 'model_response_nan_count', 'model_response_nan_rate',
        'model_response_pole_total', 'model_response_pole_nan_count', 'model_response_pole_nan_rate',
    ]
    styled = (
        df_out[display_cols].style
        .format({
            'model_response_nan_rate': '{:.1f}%',
            'model_response_pole_nan_rate': '{:.1f}%',
        }, na_rep='—')
        .set_caption('Overall NaN rates per model (sorted by model_response NaN rate desc)')
    )
    display(styled)


overall_by_model(df_detail)

In [ ]:
df_detail[df_detail['model_name'] == 'drozado_mistralai_Mixtral-8x7B-Instruct-v0.1-11d06fa6']

In [ ]:
model_name = 'drozado/mistralai/Mixtral-8x7B-Instruct-v0.1-11d06fa6'.replace('/', '_')
df_detail[df_detail['model_name'] == model_name].groupby(['experiment_name', 'experiment_type'])['nan_rate'].mean().reset_index()

## Overall NaN rate per experiment

In [ ]:
def overall_by_experiment(df_detail):
    rows = []
    for (exp_name, exp_type), grp in df_detail.groupby(['experiment_name', 'experiment_type']):
        row = {'experiment_name': exp_name, 'experiment_type': exp_type}
        for col in ['model_response', 'model_response_pole']:
            sub = grp[grp['column'] == col]
            if sub.empty:
                row[f'{col}_total'] = 0
                row[f'{col}_nan_count'] = 0
                row[f'{col}_nan_rate'] = np.nan
            else:
                total = sub['total_rows'].sum()
                nan_count = sub['nan_count'].sum()
                row[f'{col}_total'] = total
                row[f'{col}_nan_count'] = nan_count
                row[f'{col}_nan_rate'] = nan_count / total * 100 if total > 0 else np.nan
        rows.append(row)

    df_out = (
        pd.DataFrame(rows)
        .set_index(['experiment_name', 'experiment_type'])
        .sort_values('model_response_nan_rate', ascending=False)
    )
    display_cols = [
        'model_response_total', 'model_response_nan_count', 'model_response_nan_rate',
        'model_response_pole_total', 'model_response_pole_nan_count', 'model_response_pole_nan_rate',
    ]
    styled = (
        df_out[display_cols].style
        .format({
            'model_response_nan_rate': '{:.1f}%',
            'model_response_pole_nan_rate': '{:.1f}%',
        }, na_rep='—')
        .set_caption('Overall NaN rates per experiment (sorted by model_response NaN rate desc)')
    )
    display(styled)


overall_by_experiment(df_detail)